In [1]:
import numpy as np 
import csv

In [2]:
# =======================
# 1. LEITURA DO csv
# =======================

def load_data(filename):
    X = []
    y = []

    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            # Target
            survived = row["survived"]
            if survived == "":
                continue
            y.append(int(survived))

            # Features selecionadas
            # Vamos usar:
            # pclass, sex, age, sibsp, parch, fare

            pclass = float(row["pclass"]) if row["pclass"] != "" else 0

            # Convertendo sexo
            sex = 1 if row["sex"] == "female" else 0

            age = float(row["age"]) if row["age"] != "" else -1

            sibsp = float(row["sibsp"]) if row["sibsp"] != "" else 0
            parch = float(row["parch"]) if row["parch"] != "" else 0
            fare = float(row["fare"]) if row["fare"] != "" else 0

            X.append([pclass, sex, age, sibsp, parch, fare])

    return np.array(X), np.array(y).reshape(-1, 1)        

In [3]:
# =======================
# 2. TRATAMENTO DE DADOS
# =======================

def fill_missing_age(X):
    ages = X[:, 2]
    mean_age = np.mean(ages[ages != -1])

    X[:, 2] = np.where(ages == -1, mean_age, ages)
    return X


def normalize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / (std + 1e-8)


def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))

In [4]:
# =======================
# 3. FUNÇÕES DO MODELO
# =======================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_cost(X, y, beta):
    m = len(y)
    h = sigmoid(X @ beta)
    epsilon = 1e-8

    cost = -(1/m) * np.sum(
        y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon)
    )

    return cost

def gradient_descent(X, y, beta, lr, epochs):
    m = len(y)

    for i in range(epochs):
        h = sigmoid(X @ beta)
        gradient = (1/m) * (X.T @ (h - y))

        beta = beta - lr * gradient

        if i % 100 == 0:
            print(f"Epoch {i} - Cost: {compute_cost(X, y, beta):.4f}")

    return beta        

In [5]:
# =======================
# 4. TREINAMENTO
# =======================

def train(X, y):
    X = fill_missing_age(X)
    X = normalize(X)
    X = add_bias(X)

    beta = np.zeros((X.shape[1], 1))

    beta = gradient_descent(X, y, beta, lr=0.1, epochs=1000)

    return beta, X

In [6]:
# =======================
# 5. PREDIÇÃO
# =======================

def predict(X, beta):
    probs = sigmoid(X @ beta)
    return (probs >= 0.5).astype(int)

def confusion_matrix(y_true, y_pred):
    # Achatar os arrays para garantir que tenham apenas 1 dimensão
    y_t = y_true.flatten()
    y_p = y_pred.flatten()

    # O operador '&' faz a comparação bit a bit (element-wise) nos arrays do numpy
    # Faz AND lógico, elemento por elemento de cada array.
    # Por exemplo:
    # y_t = np.array([1, 0, 1, 1])
    # cond1 = (y_t == 1)
    # Resultado: [True, False, True, True]
    VP = np.sum((y_t == 1) & (y_p == 1))  # Verdadeiros Positivos
    VN = np.sum((y_t == 0) & (y_p == 0))  # Verdadeiros Negativos
    FP = np.sum((y_t == 0) & (y_p == 1))  # Falsos Positivos
    FN = np.sum((y_t == 1) & (y_p == 0))  # Falsos Negativos

    return VP, VN, FP, FN


def calculate_metrics(VP, VN, FP, FN):
    accuracy = (VP + VN) / (VP + VN + FP + FN)
    precision = VP / (VP + FP) 
    recall = VP / (VP + FN) 
    f1_score = 2 * (precision * recall) / (precision + recall) 

    return accuracy, precision, recall, f1_score

In [7]:
# =======================
# 6. SPLIT TREINO / TESTE
# =======================

def train_test_split(X, y, test_size=0.2):
    np.random.seed(42)

    indices = np.arange(len(X))
    np.random.shuffle(indices)

    split = int(len(X) * (1 - test_size))

    train_idx = indices[:split]
    test_idx = indices[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [ ]:
# =======================
# 7. EXECUÇÃO
# =======================

X, y = load_data("titanic.csv")

X_train, X_test, y_train, y_test = train_test_split(X, y)

beta, X_train_processed = train(X_train, y_train)

# Processar teste com MESMAS transformações
X_test = fill_missing_age(X_test)
X_test = normalize(X_test)
X_test = add_bias(X_test)

# Vetor de Predições
y_pred = predict(X_test, beta)

# Métricas da Matriz de Confusão
VP, VN, FP, FN = confusion_matrix(y_test, y_pred)

print("\n" + "="*30)
print(" MATRIZ DE CONFUSÃO")
print("="*30)
print(f"Verdadeiros Positivos (VP): {VP}")
print(f"Verdadeiros Negativos (VN): {VN}")
print(f"Falsos Positivos (FP): {FP}")
print(f"Falsos Negativos (FN): {FN}")

# 2. Calcular e exibir as Métricas
accuracy, precision, recall, f1_score = calculate_metrics(VP, VN, FP, FN)

print("\n" + "="*30)
print(" MÉTRICAS DE AVALIAÇÃO")
print("="*30)
print(f"Acurácia:           {accuracy:.4f} ({(accuracy*100):.1f}%)")
print(f"Precisão:           {precision:.4f} ({(precision*100):.1f}%)")
print(f"Sensibilidade:      {recall:.4f} ({(recall*100):.1f}%)")
print(f"Pontuação F1:       {f1_score:.4f} ({(f1_score*100):.1f}%)")

Epoch 0 - Cost: 0.6802
Epoch 100 - Cost: 0.4584
Epoch 200 - Cost: 0.4497
Epoch 300 - Cost: 0.4483
Epoch 400 - Cost: 0.4480
Epoch 500 - Cost: 0.4479
Epoch 600 - Cost: 0.4478
Epoch 700 - Cost: 0.4478
Epoch 800 - Cost: 0.4478
Epoch 900 - Cost: 0.4478

 MATRIZ DE CONFUSÃO
Verdadeiros Positivos (VP): 68
Verdadeiros Negativos (VN): 125
Falsos Positivos (FP): 30
Falsos Negativos (FN): 39

 MÉTRICAS DE AVALIAÇÃO
Acurácia:           0.7366 (73.7%)
Precisão:           0.6939 (69.4%)
Sensibilidade:      0.6355 (63.6%)
Pontuação F1:       0.6634 (66.3%)
